In [1]:
!pip install -q sentence-transformers datasets transformers torch

# Milestone 2

## **Introduction to Hugging Face transformers and datasets**

#### Q1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here. 

Ans : 614

In [2]:
from datasets import load_dataset

dataset = load_dataset('csv',data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')['train']

def add_combined(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

dataset = dataset.map(add_combined)
print(len(dataset[51]['combined_text']))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


#### Q2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

Ans: 30522

In [3]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


#### Q3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

Ans: 102

In [4]:
print(tokenizer.sep_token_id)

102


#### Q4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

Ans: [ 2000 , 128 ]

In [5]:
import torch

prompts = [str(x) for x in dataset['prompt']]
encoded = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
print(encoded['input_ids'].shape)

torch.Size([2000, 128])


## **BERT/RoBERTa Architecture & Attention Mechanisms**

#### Q5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

Ans: 64

In [6]:
768 / 12

64.0

#### Q6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. What is the exact shape of the last_hidden_state tensor returned? Note: We follow zero-indexing here.

Ans: [1, 31, 768]

In [7]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model= AutoModel.from_pretrained('bert-base-uncased')
model.eval()

prompt = dataset[0]['prompt']
inputs = tokenizer(prompt, return_tensors='pt')

import torch
with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


#### Q7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

Ans: -1.2001

In [8]:
cls_vector = outputs.last_hidden_state[0,0,:] 
print(round(cls_vector[:5].sum().item(), 4))

-1.2001


#### Q8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

Ans: 0.1025

In [9]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model_attn(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(tokens)
fusion_idx = tokens.index('fusion')

attn = outputs.attentions[-1][0, 0, 0, fusion_idx]
print(round(attn.item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


## *Context-Aware Embeddings ***

#### Q9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

Ans: 0.7658

In [10]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row = dataset[0]
emb_prompt = st_model.encode(row['prompt'])
emb_b = st_model.encode(row['B'])

score = util.cos_sim(emb_prompt, emb_b)
print(round(score.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


#### Q10. Build two complete ranking pipelines evaluating every row in train.csv.

#### Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

#### Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

#### First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

#### Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?

Ans: 564

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options = ['A', 'B', 'C', 'D', 'E']

def map_at_3(predictions, actual):
    score = 0.0
    for i, p in enumerate(predictions[:3]):
        if p == actual:
            score = 1.0 / (i + 1)
            break
    return score

In [12]:
# TF-IDF pipeline
tfidf_preds = []
for row in dataset:
    prompt = row['prompt']
    opts = [row[o] for o in options]
    corpus = [prompt] + opts
    vec = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vec[0:1], vec[1:])[0]
    ranked = [options[i] for i in np.argsort(sims)[::-1]]
    tfidf_preds.append(ranked)

In [13]:
# MiniLM pipeline
minilm_preds = []
for row in dataset:
    prompt = row['prompt']
    opts = [row[o] for o in options]
    embs =st_model.encode([prompt] + opts)
    sims = util.cos_sim(embs[0:1], embs[1:])[0].numpy()
    ranked = [options[i] for i in np.argsort(sims)[::-1]]
    minilm_preds.append(ranked)

# MAP@3 for MiniLM
answers = dataset['answer']
minilm_map = np.mean([map_at_3(minilm_preds[i], answers[i]) for i in range(len(dataset))])
print(f"MiniLM MAP@3: {round(minilm_map, 4)}")

# Count where TF-IDF misses but MiniLM hits (in top 3)
count = 0
for i in range(len(dataset)):
    ans = answers[i]
    tfidf_hit = ans in tfidf_preds[i][:3]
    minilm_hit = ans in minilm_preds[i][:3]
    if not tfidf_hit and minilm_hit:
        count += 1
print(f"MiniLM wins over TF-IDF count: {count}")

MiniLM MAP@3: 0.4231
MiniLM wins over TF-IDF count: 564


## **Zero-shot classification concepts**

#### Q11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

Ans: 0.4575

In [14]:
from transformers import pipeline

zsc = pipeline('zero-shot-classification')  # defaults to facebook/bart-large-mnli

row1 = dataset[1]
result = zsc(row1['prompt'], candidate_labels=[row1['A'], row1['B'], row1['C']])
print(result)
print(round(result['scores'][0], 4))  # top-ranked option's probability

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

#### Q12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

#### What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

Ans: 0.9995

In [15]:
result_multi = zsc(row1['prompt'], candidate_labels=[row1['A'], row1['B'], row1['C']], multi_label=True)

sum_softmax = sum(result['scores'])
sum_sigmoid = sum(result_multi['scores'])
print(round(abs(sum_softmax - sum_sigmoid), 4))

0.9995


#### Q13. Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

Ans: B

In [18]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')
model.eval()

row0 = dataset[0]
input_str = f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."

inputs = tokenizer(input_str, return_tensors='pt')
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=5)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


B
